In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm

In [2]:
alpha = 0.95
k = norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(k) 

2.0627128075074257


In [3]:
df = pd.read_csv("selected_monthly_return.csv", encoding="utf-8-sig")
tickers = df["Ticker"].tolist()
returns = df.drop(columns=["PERMNO", "Ticker", "average_monthly_return"])
mu = df["average_monthly_return"].to_numpy(dtype=float)

## Covariance Matrix

In [4]:
cov_df = pd.read_csv("covariance_matrix.csv", skiprows=1, encoding="utf-8-sig")

Sigma = cov_df.drop(columns=["PERMNO", "Ticker"]).to_numpy(dtype=float)

eigvals, eigvecs = np.linalg.eigh(Sigma)
A = np.sqrt(np.clip(eigvals, 0, None))[:, None] * eigvecs.T

In [5]:
# model
m = gp.Model("portfolio")
# weights
w = m.addMVar(len(tickers), vtype=GRB.CONTINUOUS, name="w")
# standard deviation of portfolio
s = m.addVar(vtype=GRB.CONTINUOUS, name="s")

Set parameter Username
Set parameter LicenseID to value 2866621
Academic license - for non-commercial use only - expires 2027-09-22


In [6]:
m.addConstr(w.sum() == 1, name="full_investment")
m.addConstr(w <= 0.2, name="max_weight")

<MConstr (37,) *awaiting model update*>

In [7]:
# helper variable for finding s
y = m.addMVar(37, lb=-GRB.INFINITY, name="y")
m.addConstr(y == A @ w)
# constraint to find s
m.addGenConstrNorm(s, y, 2.0)   

<gurobi.GenConstr *Awaiting Model Update*>

In [8]:
m.setObjective(-1 * w @ mu + k * s, gp.GRB.MINIMIZE)

In [9]:
m.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 PRO 5850U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 75 rows, 75 columns and 1480 nonzeros
Model fingerprint: 0x1110c7f5
Model has 1 general constraint
Variable types: 75 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [2e-03, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-01, 1e+00]
Presolve removed 37 rows and 0 columns
Presolve time: 0.01s
Presolved: 39 rows, 112 columns, 1476 nonzeros
Presolved model has 37 quadratic constraint(s)
Variable types: 112 continuous, 0 integer (0 binary)
Root relaxation presolve time: 0.02s
Root relaxation presolved: 117 rows, 186 columns, 1702 nonzeros
Root relaxation presolved model has 37 second-order cone constraints
Root barrier log...

Ordering time: 0.00